# Claude Code Installer
> Install MCP configuration for Claude Code

In [ ]:
#| default_exp install.claude

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Optional

from nbdev_mcp.install.base import (
    BaseInstaller, InstallResult, MCPServerConfig,
    which, run_command, read_json_config, write_json_config,
    get_python_path, get_claude_config_path
)

## Claude Installer

In [ ]:
#| export
class ClaudeInstaller(BaseInstaller):
    """Installer for Claude Code.
    
    Supports two installation methods:
    1. Native CLI: `claude mcp add` (preferred)
    2. Config file: Write to ~/.claude.json
    
    Attributes
    ----------
    use_cli : bool
        Prefer CLI installation if available.
    """
    
    def __init__(self, use_cli: bool = True):
        self.use_cli = use_cli
    
    @property
    def name(self) -> str:
        return 'claude'
    
    @property
    def config_path(self) -> Path:
        return get_claude_config_path()
    
    @property
    def has_cli(self) -> bool:
        """Check if Claude CLI is available."""
        return which('claude') is not None
    
    def is_configured(self) -> bool:
        """Check if nbdev is already configured."""
        config = read_json_config(self.config_path)
        return 'nbdev' in config.get('mcpServers', {})
    
    def install_via_cli(self, server: MCPServerConfig) -> bool:
        """Install using Claude CLI."""
        if not self.has_cli:
            return False
        
        args = [
            'claude', 'mcp', 'add',
            '--scope', 'user',
            '--transport', server.transport,
            server.name,
            '--',
            server.command,
            *server.args
        ]
        
        result = run_command(args)
        return result.returncode == 0
    
    def uninstall_via_cli(self) -> bool:
        """Uninstall using Claude CLI."""
        if not self.has_cli:
            return False
        
        result = run_command(['claude', 'mcp', 'remove', 'nbdev'])
        return result.returncode == 0
    
    def do_install(self, server: MCPServerConfig) -> None:
        """Perform the installation."""
        if self.use_cli and self.has_cli:
            self.uninstall_via_cli()
            if self.install_via_cli(server):
                return
        
        config = read_json_config(self.config_path)
        if 'mcpServers' not in config:
            config['mcpServers'] = {}
        
        config['mcpServers'][server.name] = server.to_claude_format()
        write_json_config(self.config_path, config)
    
    def do_uninstall(self) -> None:
        """Remove the configuration."""
        if self.use_cli and self.has_cli:
            self.uninstall_via_cli()
        
        config = read_json_config(self.config_path)
        if 'mcpServers' in config and 'nbdev' in config['mcpServers']:
            del config['mcpServers']['nbdev']
            write_json_config(self.config_path, config)

## Convenience Function

In [ ]:
#| export
def install_claude(
    python_path: Optional[str] = None,
    force: bool = False,
    dry_run: bool = False,
    use_cli: bool = True
) -> InstallResult:
    """Install nbdev-mcp configuration for Claude Code.
    
    Parameters
    ----------
    python_path : str, optional
        Python interpreter path. Defaults to current.
    force : bool
        Overwrite existing configuration.
    dry_run : bool
        Show what would be done without making changes.
    use_cli : bool
        Use `claude mcp add` CLI if available.
    
    Returns
    -------
    InstallResult
        Result of the installation.
    """
    installer = ClaudeInstaller(use_cli=use_cli)
    server = MCPServerConfig.for_nbdev(python_path)
    return installer.install(server, force=force, dry_run=dry_run)


def uninstall_claude(dry_run: bool = False, use_cli: bool = True) -> InstallResult:
    """Remove nbdev-mcp configuration from Claude Code."""
    installer = ClaudeInstaller(use_cli=use_cli)
    return installer.uninstall(dry_run=dry_run)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()